# Decision Tree Classifier — Titanic

Hands-on practice covering a baseline tree, pre-pruning, and post-pruning.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
titanic = sns.load_dataset("titanic")
titanic.head()
titanic.isnull().sum()


In [ ]:
features = ["pclass", "sex", "fare", "embarked", "age"]
target = ["survived"]


## Missing Data and Encoding

In [ ]:
from sklearn.impute import SimpleImputer

imp_median = SimpleImputer(strategy="median")
titanic[["age"]] = imp_median.fit_transform(titanic[["age"]])

imp_freq = SimpleImputer(strategy="most_frequent")
titanic[["embarked"]] = imp_freq.fit_transform(titanic[["embarked"]])


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
titanic["sex"] = le.fit_transform(titanic["sex"])
titanic["embarked"] = le.fit_transform(titanic["embarked"])


In [ ]:
X = titanic[features]
y = titanic[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


## Baseline Decision Tree — No Pruning

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("accuracy:", accuracy_score(y_test, y_pred))


## Pre-Pruning

In [ ]:
max_depths = [2, 3, 4, 5, 6, 7, 8, 9, 10]

for depth in max_depths:
    model = DecisionTreeClassifier(max_depth=depth)
    model.fit(X_train, y_train)
    print(f"for depth={depth}, accuracy={model.score(X_test, y_test)}")


In [ ]:
min_samples_splits = [5, 10, 15, 20, 25, 30]

for split in min_samples_splits:
    model = DecisionTreeClassifier(max_depth=4, min_samples_split=split)
    model.fit(X_train, y_train)
    print(f"for sample split={split}, accuracy={model.score(X_test, y_test)}")


## Post-Pruning — Cost-Complexity Pruning

In [ ]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)

path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

trees = []
for alpha in ccp_alphas:
    tree = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    tree.fit(X_train, y_train)
    trees.append((tree, alpha))


In [ ]:
best_acc = 0
best_alpha = 0

for tree, alpha in trees:
    current_accuracy = tree.score(X_test, y_test)
    if current_accuracy > best_acc:
        best_acc = current_accuracy
        best_alpha = alpha

best_model = DecisionTreeClassifier(ccp_alpha=best_alpha, max_depth=4)
best_model.fit(X_train, y_train)
print(best_model.score(X_test, y_test))


## Tree Visualization

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 10))
plot_tree(best_model, feature_names=X.columns, class_names=["Died", "Survived"], filled=True)
plt.tight_layout()
plt.show()
